<a href="https://colab.research.google.com/github/nisabulut/digital-image-processing/blob/main/5_3_goruntu_isleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get update
!sudo apt-get install libopencv-dev

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.6 kB]
Fetched 78.4 kB in 1s (54.1 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not

In [ ]:
%%writefile grayscale_converter.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <cmath>

// 1. Kendi Gri Tonlama Fonksiyonumuz
cv::Mat convertToGrayCustom(const cv::Mat& colorImage) {
    // Tek kanallı (CV_8UC1) boş bir matris oluşturuyoruz
    cv::Mat grayImage(colorImage.rows, colorImage.cols, CV_8UC1);

    for (int r = 0; r < colorImage.rows; ++r) {
        for (int c = 0; c < colorImage.cols; ++c) {
            // OpenCV pikselleri BGR (Mavi, Yeşil, Kırmızı) sırasıyla okur
            cv::Vec3b pixel = colorImage.at<cv::Vec3b>(r, c);
            uchar b = pixel[0];
            uchar g = pixel[1];
            uchar r_val = pixel[2];

            // Standart Gri Tonlama formülü (Luminance)
            uchar grayValue = cvRound(0.299 * r_val + 0.587 * g + 0.114 * b);
            grayImage.at<uchar>(r, c) = grayValue;
        }
    }
    return grayImage;
}

int main() {
    std::string imagePath = "image.jpg";
    cv::Mat image = cv::imread(imagePath, cv::IMREAD_COLOR);

    if (image.empty()) {
        std::cerr << "Hata: Gorsel yuklenemedi!" << std::endl;
        return -1;
    }

    // 2. OpenCV'nin hazır fonksiyonu ile dönüştürme
    cv::Mat grayCV;
    cv::cvtColor(image, grayCV, cv::COLOR_BGR2GRAY);

    // 3. Kendi fonksiyonumuz ile dönüştürme
    cv::Mat grayCustom = convertToGrayCustom(image);

    // 4. Hata Değerini (Mutlak Farkların Toplamı) Hesaplama
    // Görüntü boyutu büyük olabileceği için toplamı long long türünde tutuyoruz
    long long totalError = 0;

    for (int r = 0; r < image.rows; ++r) {
        for (int c = 0; c < image.cols; ++c) {
            int valCV = grayCV.at<uchar>(r, c);
            int valCustom = grayCustom.at<uchar>(r, c);
            totalError += std::abs(valCV - valCustom);
        }
    }

    std::cout << "Gorsel Boyutu: " << image.cols << "x" << image.rows << std::endl;
    std::cout << "Toplam Hata Degeri (Mutlak Farklarin Toplami): " << totalError << std::endl;

    cv::imwrite("gray_cv.png", grayCV);
    cv::imwrite("gray_custom.png", grayCustom);

    return 0;
}

Overwriting grayscale_converter.cpp


In [ ]:
!g++ grayscale_converter.cpp -o grayscale_converter `pkg-config --cflags --libs opencv4`
!./grayscale_converter

Gorsel Boyutu: 654x520
Toplam Hata Degeri (Mutlak Farklarin Toplami): 66
